In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
import torch
from astropy.io import fits

parent_dir = os.path.abspath(os.path.join(os.path.dirname('utils.py'), '..'))
sys.path.append(parent_dir)
import utils as ut  # noqa: E402

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUT_FEATURES = ['FLAG']
# BANDS = ['f150w', 'f277w', 'f444w']
BANDS = ['f444w-f150w', 'f444w-f277w']
SEED = 42
BEST_ACC = 0

# HYPERPARAMETERS
INPUT_SIZE = 67
EPOCHS = 200
NUM_TRIALS = 20
PATIENCE = 20
OPTIMIZE = True
SCALING = 'pytorch_normalize'

if OPTIMIZE:
    COMMENT = input('Enter OPTIMIZATION comment: ')

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

if SEED:
    ut.set_seeds(SEED)

## Prepare Data

In [ ]:
cwd = Path.cwd()
cutout_dir = cwd.parent / 'data' / 'cutouts'
model_dir = cwd.parent / 'models'

In [ ]:
# LOAD DATA INTO DF

cols = {'idx': [], 'images': [], 'header': []}

for folder in cutout_dir.iterdir():
    if folder.is_dir():
        cols['idx'].append(folder.stem)

        temp_df = ut.fits_df(folder)
        temp_df.sort_values(by='BAND')

        # load images
        temp_imgs = {}
        for _, row in temp_df.iterrows():
            hdul = fits.open(row['FILEPATH'])
            data = hdul[0].data.astype('float32')
            header = hdul[0].header
            temp_imgs[row['BAND']] = data
            hdul.close()
        cols['images'].append(temp_imgs)
        cols['header'].append(header)

# turn dict into df and expand header columns
df = pd.DataFrame(cols)
to_expand = ['header', 'images']
for col in to_expand:
    temp_df = pd.DataFrame(df[col].tolist(), index=df.index)
    df = df.join(temp_df)

df.drop(columns=['header', 'BAND', 'images'], inplace=True)

# shuffle
df = df.sample(frac=1, random_state=SEED) if SEED else df.sample(frac=1)

In [ ]:
# GET SUBTRACTED AND DIVIDED BANDS

numers = ['f150w', 'f277w']
denominator = 'f444w'

for numerator in numers:
    df[numerator + '/' + denominator] = df[numerator] / df[denominator]
    df[denominator + '-' + numerator] = df[denominator] - df[numerator]